# Difference Finite Scheme derived from MRT Lattice Boltzmann for Convective-Diffusive Equation

Based on the works of {cite:t}`bellotti2022finite` and {cite:t}`chen2023fourth`, we will develop a difference finite description of lattice Boltzmann equation in the multi-relaxation-times (MRT) framework.

## MRT Lattice Boltzmann equation for Convective-Diffusive Equation

To describe convective-diffusive problems, the MRT lattice Boltzmann formulation is described by:

$$
f_i( x_{\alpha} + e_{i,\alpha} \delta t, t+\delta t) = f_i(x_{\alpha}, t)  - (\textbf{M}^{-1}\textbf{S}\textbf{M})_{ik}\left( f_{k} - f_{k}^{eq}  \right), 
$$(MRT-LB-conv-df-Eq)

where $\textbf{M}$ is matrix of moments, $\textbf{S}$ is the diagonal matrix of relaxation times, and $\textbf{M}^{-1}$ is the inverse matrix of $\textbf{M}$. Considering the problem description in a lattice D1Q3, we have:

$$
\textbf{M} =\begin{pmatrix} 1 \\ e_{i,x} \\ 3e_{i,x}^{2}-2 \end{pmatrix} =\begin{pmatrix} 1 & 1 & 1 \\ 0 & 1   & -1 \\ -2  & 1 & 1 \end{pmatrix},  \qquad \qquad  \textbf{S} = \begin{pmatrix} s_0 & 0   & 0 \\ 0   & s_1 & 0 \\ 0   & 0   & s_2 \end{pmatrix},  \qquad \qquad \textbf{M}^{-1}=\begin{pmatrix}\dfrac{1}{3} & 0 & -\dfrac{1}{3} \\ \dfrac{1}{3} & \dfrac{1}{2} & \dfrac{1}{6} \\ \dfrac{1}{3} & -\dfrac{1}{2} & \dfrac{1}{6} \end{pmatrix}.
$$

The equilibrium distribution function is defined by

$$
f^{eq}_{i} = w_{i}\left( u \right).
$$


In [1]:
import warnings
warnings.filterwarnings("ignore")
from pylab import *
from __future__ import division
from sympy import *
import numpy as np
from sympy import S, collect, expand, factor, Wild
from sympy import fraction, Rational, Symbol
from sympy import symbols, sqrt, Rational
import sympy as sp
from IPython.display import display, Math, Latex
#-------------------------------------------------Símbolos----------------------------------------------
omega, u, B, w, C, Lamb = symbols('omega, u, B_{\\alpha}, w, C, \\Lambda_{\\alpha\\beta}')
wi, cx, cy, cs = symbols('w_{i} c_{x} c_{y} c_{s}')
fi, f0, f1, f2  = symbols('f_{i} f_{0} f_{1} f_{2}')
#-------------------------------------------------Funções----------------------------------------------
feq = Function('feq')(wi, cx, cy)
fneq = Function('fneq')(wi, cx, cy)
f = Function('f')(fi)
#----------------------------------------------Lattice-D2Q9---Variáveis----------------------------------------------
fi=np.array([f0,f1,f2])
w0=Rational(4,6);w1=Rational(1,6)
wi=np.array([w0,w1,w1])
cx=np.array([0,1,-1])
as2=Rational(3)
cs2=1/as2
#-------------------------------------------------Calc.Func------------------------------------------------
f= fi
feq=wi*(u)

The different orders of equilibrium moments can be recovered by:

In [3]:
a0=simplify(sum(feq))
ax=simplify(sum(feq*cx))
axx=simplify(sum(feq*cx*cx))
axxx=simplify(sum(feq*cx*cx*cx))
display(Math(r"\underbrace{\sum_{i=0} f_{i}^{eq} =\sum_{i=0} f_{i} }_{\textrm{Zero-Order Moment}} =" +  sp.latex(a0) 
            +r",\quad \quad \underbrace{\sum_{i=0} f_{i}^{eq}e_{i,x} }_{\textrm{x-First-Order Moment}} =" +  sp.latex(ax)
            +r",\quad \quad \underbrace{\sum_{i=0} f_{i}^{eq} e_{i,x}e_{i,x} }_{\textrm{xx-Second-Order Moment}} =" +  sp.latex(axx)
            +r",\quad \quad \underbrace{\sum_{i=0} f_{i}^{eq} e_{i,x}e_{i,x}e_{i,x} }_{\textrm{xxx-Third-Order Moment}} =" +  sp.latex(axxx) ))

aH0=simplify(sum(feq))
aHx=simplify(sum(feq*cx))
aHxx=simplify(sum(feq*(cx*cx-cs2)))
aHxxx=simplify(sum(feq*(cx*cx-3*cs2)*cx))
display(Math(r"\underbrace{\sum_{i=0} f_{i}^{eq} =\sum_{i=0} f_{i} }_{\textrm{Zero-Order Hermite Moment}} =" +  sp.latex(aH0) 
            +r",\quad \quad \underbrace{\sum_{i=0} f_{i}^{eq}e_{i,x} }_{\textrm{x-First-Order Hermite Moment}} =" +  sp.latex(aHx)
            +r",\quad \quad \underbrace{\sum_{i=0} f_{i}^{eq} \left(e_{i,x}e_{i,x} - \frac{1}{a_{s}^{2}}\right) }_{\textrm{xx-Second-Order Hermite Moment}} =" +  sp.latex(aHxx)
            +r",\quad \quad \underbrace{\sum_{i=0} f_{i}^{eq} \left(e_{i,x}e_{i,x} - \frac{3}{a_{s}^{2}}\right) e_{i,x} }_{\textrm{xxx-Third-Order Hermite Moment}} =" +  sp.latex(aHxxx) ))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## FDM Interpretation in MRT Framework

Reinterpreting the Eq. {eq}`MRT-LB-conv-df-Eq` with a shift in distribution function direction:

$$
f_{i,x}^{t+1}  = f_{i,x-e_{i}}^{t}  - (\textbf{M}^{-1}\textbf{S}\textbf{M})_{ik}\left( f_{k,x-e_{i}}^{t} - f_{k,x-e_{i}}^{eq,t} \right) =   (\textbf{M}^{-1} (\textbf{I} -\textbf{S}) \textbf{M})_{ik} f_{k,x-e_{i}}^{t} + (\textbf{M}^{-1}\textbf{S}\textbf{M})_{ik} f_{k,x-e_{i}}^{eq,t}  , 
$$

where $f_i( x_{\alpha} + e_{i,\alpha} \delta t, t+\delta t)=f_{i,x+c_{i}}^{t+1}$ for simplificate the notation and $\textbf{I}$ is identity matrix. Defining the shift operator $T_{\Delta x}^{c_{i}}$, where $f_{k,x-e_{i}}^{t}= T_{\Delta x}^{c_{i}}[f_{k,x}^{t}]$, we can rewrite the above equation inside of the moments space:

$$
\textbf{m}_{k,x}^{t+1}  = \textbf{P}\textbf{m}_{k,x}^{t}  + \textbf{Q}\textbf{m}_{k,x}^{eq,t}, 
$$(preM-RT-LB-conv-df-Eq)

where $\textbf{m}_{k,x}^{t}=M_{ik}f_{i,x}^{t}$, $k$ is the index of moments in the space moment, $\textbf{T}:= \textbf{M} (diag\left( T_{\Delta x}^{c_{0}}, T_{\Delta x}^{c_{1}}, T_{\Delta x}^{c_{2}} \right)) \textbf{M}^{-1}$, $\textbf{P}:=\textbf{T}(\textbf{I}-\textbf{S})$, and , $\textbf{Q}:=\textbf{T}\textbf{S}$. The matrices are given by:

In [4]:
import warnings
warnings.filterwarnings("ignore")
from pylab import *
from sympy import *
import numpy as np

fic, f0c, f1c, f2c = symbols(['f_{i\,x+c_{i}}^t','f_{0\,x+c_{i}}^t','f_{1\,x+c_{i}}^t','f_{2\,x+c_{i}}^t'])
f0tp1, f1tp1, f2tp1 = symbols(['f_{0\,x}^{t+1}','f_{1\,x}^{t+1}','f_{2\,x}^{t+1}'])
f0, f1, f2 = symbols(['f_{0\,x}^t','f_{1\,x}^t','f_{2\,x}^t'])
f0m1, f1m1, f2m1, f0p1, f1p1, f2p1 = symbols(['f_{0\,x-1}^t','f_{1\,x-1}^t','f_{2\,x-1}^t','f_{0\,x+1}^t','f_{1\,x+1}^t','f_{2\,x+1}^t'])
f0m1t1, f1m1t1, f2m1t1, f0p1t1, f1p1t1, f2p1t1 = symbols(['f_{0\,x-1}^{t-1}','f_{1\,x-1}^{t-1}','f_{2\,x-1}^{t-1}','f_{0\,x+1}^{t-1}','f_{1\,x+1}^{t-1}','f_{2\,x+1}^{t-1}'])
f0t1 = symbols('f_{0\,x}^{t-1}')
w0, w1, w2 = symbols(['w_0','w_1','w_2'])
s0, s1, s2 = symbols(['s_0','s_1','s_2'])
phi, phic, phim1, phip1 = symbols(['\\phi_{x}^{t}','\\phi_{x-c_{i}}^{t}','\\phi_{x-1}^{t}','\\phi_{x+1}^{t}'])
phitp1, phit1, phit2= symbols(['\\phi_{x}^{t+1}','\\phi_{x}^{t-1}','\\phi_{x}^{t-2}'])
phim1t1,phip1t1= symbols(['\\phi_{x-1}^{t-1}','\\phi_{x+1}^{t-1}'])
Tc0, Tc1, Tc2= symbols([r'T_{\Delta_{x}}^{c_{0}}','T_{\Delta_{x}}^{c_{1}}','T_{\Delta_{x}}^{c_{2}}'])
cx=np.array([0,1,-1])
wi=np.array([w0,w1,w2])
wiM=Matrix([w0,w1,w2])
fi=np.array([f0,f1,f2])
fM = Matrix([f0,f1,f2])
ficM = Matrix([f0c,f1c,f2c])
I=eye(3)
Tc = Matrix([I[0,:]*Tc0,I[1,:]*Tc1,I[2,:]*Tc2])
SM=Matrix([I[0,:]*s0,I[1,:]*s1,I[2,:]*s2])
feq=phi*wi
feqM=Matrix([feq[0],feq[1],feq[2]])
m0=np.array([1,1,1])
m1=cx
m2=3*cx*cx - 2
M=np.array([m0,m1,m2])
MM=Matrix([m0,m1,m2])
MMinv=MM.inv()

In [5]:
TM=sp.simplify(MM*Tc*MMinv)
P=sp.simplify(TM*(I-SM))
Q=sp.simplify(TM*SM)

display(Math(r"\textbf{T} =" +  sp.latex(TM) +"," ))
display(Math(r"\textbf{P} =" +  sp.latex(P) +"," ))
display(Math(r"\textbf{Q} =" +  sp.latex(Q) +"." ))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

By the Eq. {eq}`preM-RT-LB-conv-df-Eq` can be shifted in time, we recursively over $n$ timestep replace the shifted time function in it:

$$
\underbrace{ \textbf{m}_{k,x}^{t+1}  = \textbf{P}\textbf{m}_{k,x}^{t}  + \textbf{Q}\textbf{m}_{k,x}^{eq,t} \quad \rightarrow \quad  \textbf{m}_{k,x}^{t}  = \textbf{P}\textbf{m}_{k,x}^{t-1}  + \textbf{Q}\textbf{m}_{k,x}^{eq,t-1} }_{\textrm{Time Shift}} \qquad  \textrm{Recursive Substitution $n$ times:} \qquad \textbf{m}_{k,x}^{t+1}  = \textbf{P}^{n}\textbf{m}_{k,x}^{t-n+1}  + \displaystyle\sum_{l=0}^{n-1}\textbf{P}^{l}\textbf{Q}\textbf{m}_{k,x}^{eq,t-l}.
$$(preM-RT-LB-conv-df-Eq-s2)

The next steps follow in orde to employ the Calay-Hamilton theorem {cite:p}`bellotti2022finite`. In htis way, we need to find the characteristical polinomial $\mathcal{X}_{P}$ of the matrix $\textbf{P}$ (the definition of characteristical polinomial are presented in toogle below), given by:

```{toggle}
The characteristic polynomial of a square matrix $(P \in \mathbb{R}^{n \times n})$ (or $(\mathbb{C}^{n \times n}))$ is defined as:

$$
\boxed{\mathcal{X}_{P}(X) = \det(X I - P)} \qquad \rightarrow \qquad \mathcal{X}_{P}(X) = \sum_{i=0}^{n} X^{i}\gamma^{i} 
$$

where $\gamma_{i}$ are the polynomial coefficients.

```

$$
\mathcal{X}_{P} = \gamma_{3}X^{3} +  \gamma_{2}X^{2} +  \gamma_{1}X^{1} +  \gamma_{0},
$$

where all the polynomial coefficients $\gamma_{i}$ are shifted by $T_{\Delta x}^{c_{0}}$:

In [6]:
from sympy import Matrix, symbols
X = symbols('X')
cp = P.charpoly(X)
# cp.as_expr() 
gamma3=Tc0
gamma2=sp.collect(cp.all_coeffs()[1],[s2,s1,s0])
gamma1=sp.simplify(sp.collect(cp.all_coeffs()[2],[s2*s1, s2, s1,s0]).subs(s0,0).subs(Tc1*Tc2,Tc0).subs(Tc0,1)).subs(2,2*Tc0).subs(1,Tc0)
gamma0=sp.factor(sp.collect(cp.all_coeffs()[3],[s2*s1, s2, s1,s0]).subs(s0,0).subs(Tc1*Tc2,Tc0).subs(Tc0,1))*Tc0

display(Math(r"\gamma_{3} =" +  sp.latex(gamma3) +"," ))
display(Math(r"\gamma_{2} =" +  sp.latex(gamma2) +"," ))
display(Math(r"\gamma_{1} =" +  sp.latex(gamma1) +"," ))
display(Math(r"\gamma_{0} =" +  sp.latex(gamma0) +"." ))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

To reach the above coeficient results, were considered: $T_{\Delta x}^{c_{1}}T_{\Delta x}^{c_{2}}=T_{\Delta x}^{c_{0}}$; $T_{\Delta x}^{c_{1}}T_{\Delta x}^{c_{0}}=T_{\Delta x}^{c_{1}}$; $T_{\Delta x}^{c_{2}}T_{\Delta x}^{c_{0}}=T_{\Delta x}^{c_{2}}$.Defined the characteristical polynomial, we multiply both sides of Eq. {eq}`preM-RT-LB-conv-df-Eq-s2` by $\sum_{n=0}^{q}\gamma_{n}$ (where $q$ is the number of lattice velocities) and perform a time shift by change the time variable $t$ to $\tilde{t}+n=t+1$:

$$
\begin{array}{c}
\textbf{m}_{k,x}^{t+1}  = \textbf{P}^{n}\textbf{m}_{k,x}^{t-n+1}  + \displaystyle\sum_{l=0}^{n-1}\textbf{P}^{l}\textbf{Q}\textbf{m}_{k,x}^{eq,t-l} \qquad \rightarrow \qquad \textbf{m}_{k,x}^{\tilde{t}+n}  = \textbf{P}^{n}\textbf{m}_{k,x}^{\tilde{t}}  + \displaystyle\sum_{l=0}^{n-1}\textbf{P}^{l}\textbf{Q}\textbf{m}_{k,x}^{eq,\tilde{t}+n-1-l} \qquad \rightarrow \\
\displaystyle\sum_{n=0}^{q}\gamma_{n} \textbf{m}_{k,x}^{\tilde{t}+n}  = \displaystyle\sum_{n=0}^{q}\gamma_{n}\textbf{P}^{n}\textbf{m}_{k,x}^{\tilde{t}}  + \displaystyle\sum_{n=0}^{q}\gamma_{n} \left(\displaystyle\sum_{l=0}^{n-1}\textbf{P}^{l}\textbf{Q}\textbf{m}_{k,x}^{eq,\tilde{t}+n-1-l}\right),
\end{array}
$$

by applying the Calay-Hamilton theorem, that imlpie $\sum_{n=0}^{q}\gamma_{n}\textbf{P}^{n}=0$:

$$
\displaystyle\sum_{n=0}^{q}\gamma_{n} \textbf{m}_{k,x}^{\tilde{t}+n}  = \underbrace{\left(\displaystyle\sum_{n=0}^{q}\gamma_{n}\textbf{P}^{n}\right)}_{=0}\textbf{m}_{k,x}^{\tilde{t}}  + \displaystyle\sum_{n=0}^{q}\gamma_{n}\left( \displaystyle\sum_{l=0}^{n-1}\textbf{P}^{l}\textbf{Q}\textbf{m}_{k,x}^{eq,\tilde{t}+n-1-l} \right) \qquad \rightarrow \qquad \displaystyle\sum_{n=0}^{q}\gamma_{n} \textbf{m}_{k,x}^{\tilde{t}+n}  =  \displaystyle\sum_{n=0}^{q}\gamma_{n}\left(\displaystyle\sum_{l=0}^{n-1}\textbf{P}^{l}\textbf{Q}\textbf{m}_{k,x}^{eq,\tilde{t}+n-1-l}\right),
$$

applying a second time shift by change tha variable back to $\tilde{t}+q=t+1$ and expanding the left hand-side term to isolate the time $t+1$:

$$
\begin{array}{c}
\displaystyle\sum_{n=0}^{q}\gamma_{n} \textbf{m}_{k,x}^{\tilde{t}+n}  =  \displaystyle\sum_{n=0}^{q}\gamma_{n}\left(\displaystyle\sum_{l=0}^{n-1}\textbf{P}^{l}\textbf{Q}\textbf{m}_{k,x}^{eq,\tilde{t}+n-1-l}\right)  \qquad \rightarrow \qquad \displaystyle\sum_{n=0}^{q}\gamma_{n} \textbf{m}_{k,x}^{t+1-q+n}  =  \displaystyle\sum_{n=0}^{q}\gamma_{n}\left(\displaystyle\sum_{l=0}^{n-1}\textbf{P}^{l}\textbf{Q}\textbf{m}_{k,x}^{eq,t-q+n-l}\right) \qquad \rightarrow \\
\displaystyle\sum_{n=0}^{q}\gamma_{n} \textbf{m}_{k,x}^{t+1-q+n}  =  \displaystyle\sum_{n=0}^{q}\gamma_{n}\left(\displaystyle\sum_{l=0}^{n-1}\textbf{P}^{l}\textbf{Q}\textbf{m}_{k,x}^{eq,t-q+n-l}\right) \qquad \rightarrow \qquad \gamma_{q} \textbf{m}_{k,x}^{t+1}  = -\displaystyle\sum_{n=0}^{q-1}\gamma_{n} \textbf{m}_{k,x}^{t+1-q+n} + \displaystyle\sum_{n=0}^{q}\gamma_{n}\left(\displaystyle\sum_{l=0}^{n-1}\textbf{P}^{l}\textbf{Q}\textbf{m}_{k,x}^{eq,t-q+n-l}\right),
\end{array}
$$

the last sum term can be rewritten by

$$
\gamma_{q} \textbf{m}_{k,x}^{t+1}  = -\displaystyle\sum_{n=0}^{q-1}\gamma_{n} \textbf{m}_{k,x}^{t+1-q+n} + \displaystyle\sum_{n=0}^{q}\gamma_{n}\left(\displaystyle\sum_{l=0}^{n-1}\textbf{P}^{l}\textbf{Q}\textbf{m}_{k,x}^{eq,t-q+n-l}\right) \qquad \rightarrow \qquad \gamma_{q} \textbf{m}_{k,x}^{t+1}  = -\displaystyle\sum_{n=0}^{q-1}\gamma_{n} \textbf{m}_{k,x}^{t+1-q+n} + \displaystyle\sum_{n=0}^{q-1}\left(\displaystyle\sum_{l=0}^{n}\gamma_{q+l-n}\textbf{P}^{l}\right)\textbf{Q}\textbf{m}_{k,x}^{eq,t-n}.
$$

In [7]:
import sympy as sp
# ---------------- indices / parameters ----------------
q = sp.Symbol('q', integer=True, positive=True)
n, l = sp.symbols('n l', integer=True, nonnegative=True)
m1s = sp.Symbol('m^{t+1}_{k,x}')
m0s = sp.Symbol('m^{t}_{k,x}')
mm1s = sp.Symbol('m^{t-1}_{k,x}')
mm2s = sp.Symbol('m^{t-2}_{k,x}')
meq0s = sp.Symbol('m^{eq,t}_{k,x}')
meqm1s = sp.Symbol('m^{eq,t-1}_{k,x}')
meqm2s = sp.Symbol('m^{eq,t-2}_{k,x}')

# gamma_j as an indexed coefficient sequence
gammas = sp.IndexedBase('gamma')   # gamma[j] means γ_j

# Abstract time-shifted moments:
# m(s)    := m_{k,x}^{t+s}
# meq(s)  := m_{k,x}^{eq, t+s}
ms   = sp.Function('m')
meqs = sp.Function('m_eq')

# Operators/matrices (leave as Symbols unless you need matrix algebra)
Ps = sp.Symbol('P')
Qs = sp.Symbol('Q')

# ---------------- build the equation ----------------
lhs = gammas[q] * ms(1)  # γ_q * m^{t+1}

rhs1 = -sp.summation(gammas[n] * ms(1 - q + n), (n, 0, q-1))

inner = sp.summation(gammas[q + l - n] * Ps**l, (l, 0, n))
rhs2  = sp.summation(inner * Qs * meqs(-n), (n, 0, q-1))

# eq = sp.Eq(lhs, rhs1 + rhs2)
eq = rhs1 + rhs2
eq1 = lhs.subs({ms(1):m1s,gammas[q]:gammas[3]})

# ---------------- expand for a concrete q ----------------
# SymPy can only 'doit' finite sums once q is a number.
q_val = 3  # <-- change to 2,3,4,... as needed
eq_expanded = sp.expand(eq.subs(q, q_val).doit())

# print(f"\nExpanded for q={q_val}:")
# display(eq_expanded)

eqsubs=eq_expanded.subs({ms(1):m1s,ms(0):m0s,ms(-1):mm1s,ms(-2):mm2s,meqs(0):meq0s,meqs(0):meq0s,meqs(-1):meqm1s,meqs(-2):meqm2s})
sp.Eq(eq1, eqsubs)

Eq(m^{t+1}_{k,x}*gamma[3], P**2*Q*m^{eq,t-2}_{k,x}*gamma[3] + P*Q*m^{eq,t-1}_{k,x}*gamma[3] + P*Q*m^{eq,t-2}_{k,x}*gamma[2] + Q*m^{eq,t-1}_{k,x}*gamma[2] + Q*m^{eq,t-2}_{k,x}*gamma[1] + Q*m^{eq,t}_{k,x}*gamma[3] - m^{t-1}_{k,x}*gamma[1] - m^{t-2}_{k,x}*gamma[0] - m^{t}_{k,x}*gamma[2])

Considering the solution for the moment $m^{t+1}_{0,x}$, the above equation is rewritten to:

$$
m^{t+1}_{0,x} \gamma_{3} = \left[P^{2} Q m^{eq,t-2}_{k,x} \gamma_{3}\right]_{k=0} + \left[P Q m^{eq,t-1}_{k,x} \gamma_{3}\right]_{k=0} + \left[P Q m^{eq,t-2}_{k,x} \gamma_{2}\right]_{k=0} + \left[Q m^{eq,t-1}_{k,x} \gamma_{2}\right]_{k=0} + \left[Q m^{eq,t-2}_{k,x} \gamma_{1}\right]_{k=0} + \left[Q m^{eq,t}_{k,x} \gamma_{3}\right]_{k=0} - m^{t-1}_{0,x} \gamma_{1} - m^{t-2}_{0,x} \gamma_{0} - m^{t}_{0,x} \gamma_{2}
$$(Momentum-exp-term-convec-diff)

In [8]:
import sympy as sp
# ---------------- indices / parameters ----------------
meq0k11s = sp.Symbol('m^{eq,t+1}_{0,x}')
meq0k0s = sp.Symbol('m^{eq,t}_{0,x}')
meq1k0s = sp.Symbol('m^{eq,t}_{1,x}')
meq2k0s = sp.Symbol('m^{eq,t}_{2,x}')
MeqM0s=Matrix([meq0k0s,meq1k0s,meq2k0s])
meq0k1s = sp.Symbol('m^{eq,t-1}_{0,x}')
meq1k1s = sp.Symbol('m^{eq,t-1}_{1,x}')
meq2k1s = sp.Symbol('m^{eq,t-1}_{2,x}')
MeqM1s=Matrix([meq0k1s,meq1k1s,meq2k1s])
meq0k2s = sp.Symbol('m^{eq,t-2}_{0,x}')
meq1k2s = sp.Symbol('m^{eq,t-2}_{1,x}')
meq2k2s = sp.Symbol('m^{eq,t-2}_{2,x}')
MeqM2s=Matrix([meq0k2s,meq1k2s,meq2k2s])
m0k11s = sp.Symbol('m^{t+1}_{0,x}')
m0k0s = sp.Symbol('m^{t}_{0,x}')
m0k1s = sp.Symbol('m^{t-1}_{0,x}')
m0k2s = sp.Symbol('m^{t-2}_{0,x}')

In [9]:
termmeqm1=collect(simplify(expand(Q*MeqM1s*gamma2 + P*Q*MeqM1s))[0].subs(Tc0*Tc1,Tc1).subs(Tc0*Tc2,Tc2).subs(Tc1*Tc2,Tc0).subs(s0,0),[meq0k1s,meq1k1s,meq2k1s])
termmeqm1
display(Math(r"\left[ P Q m^{eq,t-1}_{k,x} \gamma_{3} \right]_{k=0} + \left[ Q m^{eq,t-1}_{k,x} \gamma_{2} \right]_{k=0} =" +  sp.latex(termmeqm1) +"," ))

<IPython.core.display.Math object>

In [10]:
# termmeqm2 = expand((P*Q)*MeqMs*gamma2 + Q*MeqMs*gamma1)[0]
termmeqm2 = expand( ((P*P)*Q)*MeqM2s*gamma3 + (P*Q)*MeqM2s*gamma2 + Q*MeqM2s*gamma1)[0].subs(Tc1*Tc2,Tc0).subs(Tc0,1).subs(s0,0)
display(Math(r"\left[ P^{2} Q m^{eq,t-2}_{k,x} \gamma_{3} \right]_{k=0} + \left[ PQ m^{eq,t-2}_{k,x} \gamma_{2} \right]_{k=0} + \left[ Q m^{eq,t-2}_{k,x} \gamma_{1} \right]_{k=0} =" +  sp.latex(termmeqm2) +"," ))

<IPython.core.display.Math object>

In [11]:
termmeqm3 = (Q*MeqM0s)[0].subs(s0,0)
# sp.Eq(Qs*meq0s*gammas[3], termmeqm3)
display(Math(r"\left[Q m^{eq,t}_{k,x} \gamma_{3} \right]_{k=0} =" +  sp.latex(termmeqm3) +"." ))

<IPython.core.display.Math object>

Replacing the terms in the Eq. {eq}`Momentum-exp-term-convec-diff`:

In [12]:
eqsubs2=(eqsubs.subs({meqm2s:0,Ps*Qs*meqm1s*gammas[3] + Qs*meqm1s*gammas[2]:termmeqm1,Qs*meq0s*gammas[3]:termmeqm3})
        .subs({m0s:m0k0s,mm1s:m0k1s,mm2s:m0k2s}).subs({gammas[3]:gamma3,gammas[2]:gamma2,gammas[1]:gamma1,gammas[0]:gamma0}).subs({s0:0})
        .subs({expand((Tc1*s1*s2-Tc1*s1-Tc2*s1*s2+Tc2*s1)/2):factor(expand((Tc1*s1*s2-Tc1*s1-Tc2*s1*s2+Tc2*s1)/2))})
        .subs({expand((2*Tc0*s1*s2-2*Tc0*s2-Tc1*s1*s2+Tc1*s2-Tc2*s1*s2+Tc2*s2)/6):factor(expand((2*Tc0*s1*s2-2*Tc0*s2-Tc1*s1*s2+Tc1*s2-Tc2*s1*s2+Tc2*s2)/6))}) )
sp.Eq(eq1.subs({m1s:m0k11s,gammas[3]:gamma3}), eqsubs2)

Eq(T_{\Delta_{x}}^{c_{0}}*m^{t+1}_{0,x}, T_{\Delta_{x}}^{c_{0}}*m^{t-2}_{0,x}*(s_1 - 1)*(s_2 - 1) + m^{eq,t-1}_{1,x}*s_1*(T_{\Delta_{x}}^{c_{1}} - T_{\Delta_{x}}^{c_{2}})*(s_2 - 1)/2 + m^{eq,t-1}_{2,x}*s_2*(s_1 - 1)*(2*T_{\Delta_{x}}^{c_{0}} - T_{\Delta_{x}}^{c_{1}} - T_{\Delta_{x}}^{c_{2}})/6 + m^{eq,t}_{1,x}*s_1*(T_{\Delta_{x}}^{c_{1}} - T_{\Delta_{x}}^{c_{2}})/2 + m^{eq,t}_{2,x}*s_2*(-2*T_{\Delta_{x}}^{c_{0}} + T_{\Delta_{x}}^{c_{1}} + T_{\Delta_{x}}^{c_{2}})/6 - m^{t-1}_{0,x}*(T_{\Delta_{x}}^{c_{0}} + T_{\Delta_{x}}^{c_{1}} + T_{\Delta_{x}}^{c_{2}} + s_1*s_2*(T_{\Delta_{x}}^{c_{0}} + T_{\Delta_{x}}^{c_{1}} + T_{\Delta_{x}}^{c_{2}})/3 - s_1*(2*T_{\Delta_{x}}^{c_{0}} + T_{\Delta_{x}}^{c_{1}} + T_{\Delta_{x}}^{c_{2}})/2 - s_2*(2*T_{\Delta_{x}}^{c_{0}} + 5*T_{\Delta_{x}}^{c_{1}} + 5*T_{\Delta_{x}}^{c_{2}})/6) - m^{t}_{0,x}*(-T_{\Delta_{x}}^{c_{0}} - T_{\Delta_{x}}^{c_{1}} - T_{\Delta_{x}}^{c_{2}} + s_1*(T_{\Delta_{x}}^{c_{1}}/2 + T_{\Delta_{x}}^{c_{2}}/2) + s_2*(2*T_{\Delta_{x}}^{c_{0}

In [13]:
m0k0sdp1 = sp.Symbol('m^{t}_{0,x+1}')
m0k0sdm1 = sp.Symbol('m^{t}_{0,x-1}')
m0k1sdp1 = sp.Symbol('m^{t-1}_{0,x+1}')
m0k1sdm1 = sp.Symbol('m^{t-1}_{0,x-1}')

meq1k0sdp1 = sp.Symbol('m^{eq,t}_{1,x+1}')
meq1k0sdm1 = sp.Symbol('m^{eq,t}_{1,x-1}')
meq2k0sdp1 = sp.Symbol('m^{eq,t}_{2,x+1}')
meq2k0sdm1 = sp.Symbol('m^{eq,t}_{2,x-1}')

meq1k1sdp1 = sp.Symbol('m^{eq,t-1}_{1,x+1}')
meq1k1sdm1 = sp.Symbol('m^{eq,t-1}_{1,x-1}')
meq2k1sdp1 = sp.Symbol('m^{eq,t-1}_{2,x+1}')
meq2k1sdm1 = sp.Symbol('m^{eq,t-1}_{2,x-1}')

In [14]:
eqsubs3 = (collect(expand(eqsubs2).subs({Tc0:1})
                .subs({Tc2*m0k0s:m0k0sdp1,Tc1*m0k0s:m0k0sdm1})
                .subs({Tc2*m0k1s:m0k1sdp1,Tc1*m0k1s:m0k1sdm1})
                .subs({Tc2*meq1k0s:meq1k0sdp1,Tc1*meq1k0s:meq1k0sdm1})
                .subs({Tc2*meq2k0s:meq2k0sdp1,Tc1*meq2k0s:meq2k0sdm1})
                .subs({Tc2*meq1k1s:meq1k1sdp1,Tc1*meq1k1s:meq1k1sdm1})
                .subs({Tc2*meq2k1s:meq2k1sdp1,Tc1*meq2k1s:meq2k1sdm1})
                ,[m0k2s, m0k0sdp1,m0k0sdm1,m0k0s, m0k1sdm1,m0k1sdp1,m0k1s, meq1k0sdm1,meq1k0sdp1,meq1k0s, meq2k0sdm1,meq2k0sdp1,meq2k0s,
                 meq1k1sdm1,meq1k1sdp1,meq1k1s, meq2k1sdm1,meq2k1sdp1,meq2k1s])
           .subs({(s1*s2-s1-s2+1):factor(s1*s2-s1-s2+1)}) 
          )
sp.Eq(eq1.subs({m1s:m0k11s,gammas[3]:1}), eqsubs3)

Eq(m^{t+1}_{0,x}, m^{eq,t-1}_{1,x+1}*(-s_1*s_2/2 + s_1/2) + m^{eq,t-1}_{1,x-1}*(s_1*s_2/2 - s_1/2) + m^{eq,t-1}_{2,x+1}*(-s_1*s_2/6 + s_2/6) + m^{eq,t-1}_{2,x-1}*(-s_1*s_2/6 + s_2/6) + m^{eq,t-1}_{2,x}*(s_1*s_2/3 - s_2/3) - m^{eq,t}_{1,x+1}*s_1/2 + m^{eq,t}_{1,x-1}*s_1/2 + m^{eq,t}_{2,x+1}*s_2/6 + m^{eq,t}_{2,x-1}*s_2/6 - m^{eq,t}_{2,x}*s_2/3 + m^{t-1}_{0,x+1}*(-s_1*s_2/3 + s_1/2 + 5*s_2/6 - 1) + m^{t-1}_{0,x-1}*(-s_1*s_2/3 + s_1/2 + 5*s_2/6 - 1) + m^{t-1}_{0,x}*(-s_1*s_2/3 + s_1 + s_2/3 - 1) + m^{t-2}_{0,x}*(s_1 - 1)*(s_2 - 1) + m^{t}_{0,x+1}*(-s_1/2 - s_2/6 + 1) + m^{t}_{0,x-1}*(-s_1/2 - s_2/6 + 1) + m^{t}_{0,x}*(1 - 2*s_2/3))

Replacing the moments:

$$
m_{0,x}^{t}=u_{x}^{t}, \qquad m_{0,x}^{eq,t}=u_{x}^{t}, \qquad m_{1,x}^{eq,t}=B_{x}^{t}, \qquad \textrm{and} \qquad m_{1,x}^{eq,t}=C_{x}^{t} + 3\Lambda_{x}^{t} - 2u_{x}^{t},
$$

in the above equation:

In [15]:
u11s = sp.Symbol('u^{t+1}_{x}')
u0s = sp.Symbol('u^{t}_{x}')
u1s = sp.Symbol('u^{t-1}_{x}')
u2s = sp.Symbol('u^{t-2}_{x}')
u0sdp1 = sp.Symbol('u^{t}_{x+1}')
u0sdm1 = sp.Symbol('u^{t}_{x-1}')
u1sdp1 = sp.Symbol('u^{t-1}_{x+1}')
u1sdm1 = sp.Symbol('u^{t-1}_{x-1}')

B0sdm1 = sp.Symbol('B^{t}_{x-1}')
B0sdp1 = sp.Symbol('B^{t}_{x+1}')
B1sdm1 = sp.Symbol('B^{t-1}_{x-1}')
B1sdp1 = sp.Symbol('B^{t-1}_{x+1}')

C0s = sp.Symbol('C^{t}_{x}')
C0sdm1 = sp.Symbol('C^{t}_{x-1}')
C0sdp1 = sp.Symbol('C^{t}_{x+1}')
C1s = sp.Symbol('C^{t-1}_{x}')
C1sdm1 = sp.Symbol('C^{t-1}_{x-1}')
C1sdp1 = sp.Symbol('C^{t-1}_{x+1}')

La0s = sp.Symbol('\\Lambda^{t}_{x}')
La0sdm1 = sp.Symbol('\\Lambda^{t}_{x-1}')
La0sdp1 = sp.Symbol('\\Lambda^{t}_{x+1}')
La1s = sp.Symbol('\\Lambda^{t-1}_{x}')
La1sdm1 = sp.Symbol('\\Lambda^{t-1}_{x-1}')
La1sdp1 = sp.Symbol('\\Lambda^{t-1}_{x+1}')

In [16]:
eqsubs4 = (eqsubs3.subs({m0k0s:u0s,m0k1s:u1s,m0k2s:u2s,m0k0sdp1:u0sdp1,m0k0sdm1:u0sdm1,m0k1sdm1:u1sdm1,m0k1sdp1:u1sdp1})
          .subs({meq1k0sdp1:B0sdp1,meq1k0sdm1:B0sdm1,meq1k1sdp1:B1sdp1,meq1k1sdm1:B1sdm1})
          .subs({meq2k0sdp1:C0sdp1+3*La0sdp1-2*u0sdp1,meq2k0sdm1:C0sdm1+3*La0sdm1-2*u0sdm1,meq2k1sdp1:C1sdp1+3*La1sdp1-2*u1sdp1,meq2k1sdm1:C1sdm1+3*La1sdm1-2*u1sdm1})
          .subs({meq2k0s:C0s+3*La0s-2*u0s,meq2k1s:C1s+3*La1s-2*u1s}))
                   
sp.Eq(eq1.subs({m1s:u11s,gammas[3]:1}), eqsubs4)

Eq(u^{t+1}_{x}, B^{t-1}_{x+1}*(-s_1*s_2/2 + s_1/2) + B^{t-1}_{x-1}*(s_1*s_2/2 - s_1/2) - B^{t}_{x+1}*s_1/2 + B^{t}_{x-1}*s_1/2 + s_2*(C^{t}_{x+1} + 3*\Lambda^{t}_{x+1} - 2*u^{t}_{x+1})/6 + s_2*(C^{t}_{x-1} + 3*\Lambda^{t}_{x-1} - 2*u^{t}_{x-1})/6 - s_2*(C^{t}_{x} + 3*\Lambda^{t}_{x} - 2*u^{t}_{x})/3 + u^{t-1}_{x+1}*(-s_1*s_2/3 + s_1/2 + 5*s_2/6 - 1) + u^{t-1}_{x-1}*(-s_1*s_2/3 + s_1/2 + 5*s_2/6 - 1) + u^{t-1}_{x}*(-s_1*s_2/3 + s_1 + s_2/3 - 1) + u^{t-2}_{x}*(s_1 - 1)*(s_2 - 1) + u^{t}_{x+1}*(-s_1/2 - s_2/6 + 1) + u^{t}_{x-1}*(-s_1/2 - s_2/6 + 1) + u^{t}_{x}*(1 - 2*s_2/3) + (-s_1*s_2/6 + s_2/6)*(C^{t-1}_{x+1} + 3*\Lambda^{t-1}_{x+1} - 2*u^{t-1}_{x+1}) + (-s_1*s_2/6 + s_2/6)*(C^{t-1}_{x-1} + 3*\Lambda^{t-1}_{x-1} - 2*u^{t-1}_{x-1}) + (s_1*s_2/3 - s_2/3)*(C^{t-1}_{x} + 3*\Lambda^{t-1}_{x} - 2*u^{t-1}_{x}))

rewriting the above equation, we have:

In [17]:
alpha1 = sp.Symbol('\\alpha_{1}')
alpha2 = sp.Symbol('\\alpha_{2}')
beta1 = sp.Symbol('\\beta_{1}')
beta2 = sp.Symbol('\\beta_{2}')
eta1 = sp.Symbol('\\eta_{1}')
eta2 = sp.Symbol('\\eta_{2}')
kappa1 = sp.Symbol('\\kappa_{1}')
kappa2 = sp.Symbol('\\kappa_{2}')
gamma = sp.Symbol('\\gamma')
# eqsubs5 = eqsubs4.subs({s1/2:beta1,s1*s2/2-s1/2:beta2})
eqsubs5 = (eqsubs4.subs({-s1*s2/3+s1/2+5*s2/6-1:beta2,-s1*s2/3+s1+s2/3-1:beta1,-s1/2-s2/6+1:alpha2,1-2*s2/3:alpha1,(s1-1)*(s2-1):gamma})
                  .subs({-s1*s2/6+s2/6:kappa2,s1*s2/3-s2/3:-2*kappa2,s2/6:kappa1,-s2/3:-2*kappa1}) 
                  .subs({s1*s2/2-s1/2:eta2,s1/2:eta1}))
sp.Eq(eq1.subs({m1s:u11s,gammas[3]:1}), eqsubs5)

Eq(u^{t+1}_{x}, -B^{t-1}_{x+1}*\eta_{2} + B^{t-1}_{x-1}*\eta_{2} - B^{t}_{x+1}*\eta_{1} + B^{t}_{x-1}*\eta_{1} + \alpha_{1}*u^{t}_{x} + \alpha_{2}*u^{t}_{x+1} + \alpha_{2}*u^{t}_{x-1} + \beta_{1}*u^{t-1}_{x} + \beta_{2}*u^{t-1}_{x+1} + \beta_{2}*u^{t-1}_{x-1} + \gamma*u^{t-2}_{x} + \kappa_{1}*(C^{t}_{x+1} + 3*\Lambda^{t}_{x+1} - 2*u^{t}_{x+1}) + \kappa_{1}*(C^{t}_{x-1} + 3*\Lambda^{t}_{x-1} - 2*u^{t}_{x-1}) - 2*\kappa_{1}*(C^{t}_{x} + 3*\Lambda^{t}_{x} - 2*u^{t}_{x}) + \kappa_{2}*(C^{t-1}_{x+1} + 3*\Lambda^{t-1}_{x+1} - 2*u^{t-1}_{x+1}) + \kappa_{2}*(C^{t-1}_{x-1} + 3*\Lambda^{t-1}_{x-1} - 2*u^{t-1}_{x-1}) - 2*\kappa_{2}*(C^{t-1}_{x} + 3*\Lambda^{t-1}_{x} - 2*u^{t-1}_{x}))

where coefficients are given by:

In [18]:
display(Math(r"\alpha_{1} =" +  sp.latex(1-2*s2/3) +r",\qquad \alpha_{2}="+  sp.latex(-s1/2-s2/6+1) +r",\qquad \beta_{1}="
            +  sp.latex(-s1*s2/3+s1+s2/3-1) +r",\qquad \beta_{2}="+  sp.latex(-s1*s2/3+s1/2+5*s2/6-1) ))
display(Math(r"\eta_{1}=" +  sp.latex(s1/2) +r",\qquad \eta_{2}=" +  sp.latex(s1*s2/2-s1/2) +r",\qquad \kappa_{1}="+  sp.latex(s2/6) 
             +r",\qquad \kappa_{1}="+  sp.latex(-s1*s2/6+s2/6) +r",\qquad \gamma="+  sp.latex((s1-1)*(s2-1)) ))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Linear Diffusive Equation

Considering linear convective-diffusive equation:

$$
\partial_t u = \partial_{\alpha}\left(\nu \partial_{\alpha} u \right) 
$$

we have $C=u$. Replacing $C$ in FD scheme obtained

In [20]:
c = sp.Symbol('c_{\\alpha}')
cb = sp.Symbol('\overline{c}_{\\alpha}')
leqsubs4=sp.collect(eqsubs4.subs({B0sdp1:0,B0sdm1:0,B1sdp1:0,B1sdm1:0})
                 .subs({C0s:u0s,La0s:0,C0sdp1:u0sdp1,La0sdp1:0,C0sdm1:u0sdm1,La0sdm1:0,
                        C1s:u1s,La1s:0,C1sdp1:u1sdp1,La1sdp1:0,C1sdm1:u1sdm1,La1sdm1:0}) , [u0s,u1s,u0sdp1,u0sdm1,u1sdp1,u1sdm1])
sp.Eq(eq1.subs({m1s:u11s,gammas[3]:1}), leqsubs4)

Eq(u^{t+1}_{x}, u^{t-1}_{x+1}*(-s_1*s_2/6 + s_1/2 + 2*s_2/3 - 1) + u^{t-1}_{x-1}*(-s_1*s_2/6 + s_1/2 + 2*s_2/3 - 1) + u^{t-1}_{x}*(-2*s_1*s_2/3 + s_1 + 2*s_2/3 - 1) + u^{t-2}_{x}*(s_1 - 1)*(s_2 - 1) + u^{t}_{x+1}*(-s_1/2 - s_2/3 + 1) + u^{t}_{x-1}*(-s_1/2 - s_2/3 + 1) + u^{t}_{x}*(1 - s_2/3))

In [24]:
alpha1s = sp.Symbol('\\alpha_{1}')
alpha1 = leqsubs4.coeff(u0s)
alpha2s = sp.Symbol('\\alpha_{2}')
alpha2 = leqsubs4.coeff(u0sdp1)
beta1s = sp.Symbol('\\beta_{1}')
beta1 = leqsubs4.coeff(u1s)
beta2s = sp.Symbol('\\beta_{2}')
beta2 = leqsubs4.coeff(u1sdp1)
gammass = sp.Symbol('\\gamma')
gamma = leqsubs4.coeff(u2s)

leqsubs5 = (leqsubs4.subs({beta2:beta2s,beta1:beta1s,alpha2:alpha2s,alpha1:alpha1s,gamma:gammass}) )
sp.Eq(eq1.subs({m1s:u11s,gammas[3]:1}), leqsubs5)

Eq(u^{t+1}_{x}, \alpha_{1}*u^{t}_{x} + \alpha_{2}*u^{t}_{x+1} + \alpha_{2}*u^{t}_{x-1} + \beta_{1}*u^{t-1}_{x} + \beta_{2}*u^{t-1}_{x+1} + \beta_{2}*u^{t-1}_{x-1} + \gamma*u^{t-2}_{x})

where coefficients are given by:

In [28]:
display(Math(r"\alpha_{1} =" +  sp.latex(alpha1) +r",\qquad \alpha_{2}="+  sp.latex(alpha2) + r",\qquad\beta_{1}=" +  sp.latex(beta1) 
             +r",\qquad \beta_{2}="+  sp.latex(beta2) +r",\qquad \gamma="+  sp.latex(gamma) ))

<IPython.core.display.Math object>